In [1]:
# Cell 1: Setup
!pip install transformers datasets google-genai -q

import torch
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import userdata

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Setup complete.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 2.0.1 requires anyio<4,>=3.1.0, but you have anyio 4.13.0 which is incompatible.
tensorflow-intel 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# Retrain small CodeBERT in this session for pipeline testing
# Uses the dedup parquets you upload from your laptop

from google.colab import files
print("Upload train_dedup.parquet and val_dedup.parquet from your laptop")
uploaded = files.upload()

Upload train_dedup.parquet and val_dedup.parquet from your laptop


Saving train_dedup.parquet to train_dedup.parquet
Saving val_dedup.parquet to val_dedup.parquet


In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

print("Loading data...")
train_df = pd.read_parquet("train_dedup.parquet").sample(5000, random_state=42)
val_df   = pd.read_parquet("val_dedup.parquet").sample(500, random_state=42)

print(f"Train: {len(train_df)} rows, balance: {train_df['label'].value_counts(normalize=True).round(3).to_dict()}")
print(f"Val:   {len(val_df)} rows")

MODEL_NAME = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok(batch):
    return tokenizer(batch["code"], padding="max_length", truncation=True, max_length=256)

train_ds = Dataset.from_pandas(train_df).map(tok, batched=True).rename_column("label", "labels")
val_ds   = Dataset.from_pandas(val_df).map(tok, batched=True).rename_column("label", "labels")
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("Loading CodeBERT...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir="pipeline_model",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
print("Training...")
trainer.train()
print("Done. Model saved to pipeline_model/")

Loading data...
Train: 5000 rows, balance: {1: 0.579, 0: 0.421}
Val:   500 rows


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading CodeBERT...


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

... (2 outputs left out)

Training...


Epoch,Training Loss,Validation Loss
1,0.000151,0.000065


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done. Model saved to pipeline_model/


In [ ]:
CHECKPOINT_PATH = "pipeline_model/checkpoint-625"

# Find actual checkpoint folder name
import os
checkpoints = [d for d in os.listdir("pipeline_model") if d.startswith("checkpoint")]
print("Available checkpoints:", checkpoints)
CHECKPOINT_PATH = f"pipeline_model/{checkpoints[-1]}"
print("Using:", CHECKPOINT_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT_PATH,
    output_attentions=True
)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

LABEL_MAP = {0: "SAFE", 1: "VULNERABLE"}
print("Model loaded with attention output enabled.")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Available checkpoints: ['checkpoint-625']
Using: pipeline_model/checkpoint-625


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded with attention output enabled.
Parameters: 124,647,170


In [ ]:
import numpy as np
import json

def predict_with_attention(code, top_k=10):
    inputs = tokenizer(
        code,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # Prediction
    probs = torch.softmax(outputs.logits, dim=1)
    pred_id = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred_id].item()
    prediction = LABEL_MAP[pred_id]

    # Attention extraction — last layer, average across heads, CLS row
    last_layer = outputs.attentions[-1]        # (1, 12, 256, 256)
    cls_attn = last_layer[0, :, 0, :]          # (12, 256)
    avg_attn = cls_attn.mean(dim=0)            # (256,)

    token_ids = inputs["input_ids"][0]
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    pairs = [
        (tokens[i], float(avg_attn[i]))
        for i in range(len(tokens))
        if token_ids[i].item() not in [
            tokenizer.cls_token_id,
            tokenizer.sep_token_id,
            tokenizer.pad_token_id
        ]
    ]
    pairs.sort(key=lambda x: x[1], reverse=True)

    # Clean RoBERTa token artifacts
    top_tokens = []
    for tok_str, score in pairs:
        clean = tok_str.replace("Ġ", "").replace("Ċ", "").strip()
        if len(clean) > 1:
            top_tokens.append(clean)
        if len(top_tokens) == top_k:
            break

    return {
        "prediction": prediction,
        "confidence": round(confidence, 4),
        "top_attention_tokens": top_tokens,
        "code": code
    }

# Quick test
test_code = """void copy_data(char *input) {
    char buffer[10];
    strcpy(buffer, input);
}"""

result = predict_with_attention(test_code)
print("Prediction:  ", result["prediction"])
print("Confidence:  ", result["confidence"])
print("Top tokens:  ", result["top_attention_tokens"][:8])

Prediction:   VULNERABLE
Confidence:   0.9999
Top tokens:   ['];', '10', ');', 'copy', 'char', 'str', 'data', 'input']


In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

def generate_explanation(inference_result):
    code = inference_result["code"]
    prediction = inference_result["prediction"]
    confidence = inference_result["confidence"]
    top_tokens = inference_result["top_attention_tokens"]

    prompt = f"""You are a software security expert analyzing C/C++ code.

A CodeBERT model analysed this code:

```c
{code}
```

Model output:
- Prediction: {prediction}
- Confidence: {confidence}
- Most attended tokens: {", ".join(top_tokens[:8])}

Respond ONLY with valid JSON, no markdown, no preamble:

{{
  "predicted_cwe": "CWE-XXX: name",
  "vulnerability_present": true or false,
  "evidence_tokens": ["token1", "token2", "token3"],
  "explanation": "2-3 sentences explaining what makes this code vulnerable or safe",
  "exploitation_risk": "how this could be exploited, or N/A if safe",
  "mitigation": "specific code fix, or N/A if safe"
}}"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    raw = response.text.strip()
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw": raw}

# Test
explanation = generate_explanation(result)
print(json.dumps(explanation, indent=2))

{
  "parse_error": true,
  "raw": "c\\nvoid copy_data(char *input) {\\n    char buffer[10];\\n    strncpy(buffer, input, sizeof(buffer) - 1);\\n    buffer[sizeof(buffer) - 1] = '\\\\0'; // Ensure null termination\\n}\\n"
}


In [ ]:
def run_pipeline(code, sample_id=None):
    inference = predict_with_attention(code)
    explanation = generate_explanation(inference)

    print(f"\n[{sample_id}] {inference['prediction']} ({inference['confidence']}) | CWE: {explanation.get('predicted_cwe','?')}")
    print(f"  Tokens: {inference['top_attention_tokens'][:5]}")
    print(f"  Explain: {explanation.get('explanation','')[:100]}...")

    return {
        "sample_id": sample_id,
        "code": code,
        "inference": inference,
        "explanation": explanation
    }

rq3_samples = [
    # CWE-121
    {"id":"CWE121_v1","cwe":"CWE-121","expected":"VULNERABLE","code":"void read_input(char *src) {\n    char dest[16];\n    strcpy(dest, src);\n    printf(\"%s\", dest);\n}"},
    {"id":"CWE121_v2","cwe":"CWE-121","expected":"VULNERABLE","code":"void process(char *data) {\n    char buf[8];\n    gets(buf);\n    return;\n}"},
    {"id":"CWE121_v3","cwe":"CWE-121","expected":"VULNERABLE","code":"void concat(char *s1, char *s2) {\n    char result[20];\n    strcpy(result, s1);\n    strcat(result, s2);\n}"},
    {"id":"CWE121_s1","cwe":"CWE-121","expected":"SAFE","code":"void read_safe(char *src) {\n    char dest[16];\n    strncpy(dest, src, sizeof(dest)-1);\n    dest[sizeof(dest)-1] = '\\0';\n}"},
    {"id":"CWE121_s2","cwe":"CWE-121","expected":"SAFE","code":"void read_line(FILE *fp) {\n    char buf[64];\n    if (fgets(buf, sizeof(buf), fp) != NULL) printf(\"%s\", buf);\n}"},
    # CWE-78
    {"id":"CWE78_v1","cwe":"CWE-78","expected":"VULNERABLE","code":"void run_cmd(char *input) {\n    char cmd[256];\n    sprintf(cmd, \"ping %s\", input);\n    system(cmd);\n}"},
    {"id":"CWE78_v2","cwe":"CWE-78","expected":"VULNERABLE","code":"void execute(char *arg) {\n    char buf[128];\n    strcpy(buf, arg);\n    system(buf);\n}"},
    {"id":"CWE78_v3","cwe":"CWE-78","expected":"VULNERABLE","code":"int run(char *cmd) {\n    return system(cmd);\n}"},
    {"id":"CWE78_s1","cwe":"CWE-78","expected":"SAFE","code":"void run_fixed(const char *file) {\n    const char *args[] = {\"/bin/ping\",\"-c\",\"1\",file,NULL};\n    execv(\"/bin/ping\",(char *const *)args);\n}"},
    {"id":"CWE78_s2","cwe":"CWE-78","expected":"SAFE","code":"void log_event(const char *type) {\n    const char *ok[] = {\"login\",\"logout\",\"error\"};\n    for (int i=0;i<3;i++) if (!strcmp(type,ok[i])) { printf(\"%s\\n\",type); return; }\n}"},
    # CWE-190
    {"id":"CWE190_v1","cwe":"CWE-190","expected":"VULNERABLE","code":"void alloc(int count) {\n    int size = count * sizeof(int);\n    int *data = malloc(size);\n    memset(data,0,size);\n}"},
    {"id":"CWE190_v2","cwe":"CWE-190","expected":"VULNERABLE","code":"short increment(short val) {\n    return val + 1;\n}"},
    {"id":"CWE190_v3","cwe":"CWE-190","expected":"VULNERABLE","code":"int compute(unsigned int a, unsigned int b) {\n    return (int)(a + b);\n}"},
    {"id":"CWE190_s1","cwe":"CWE-190","expected":"SAFE","code":"void alloc_safe(int count) {\n    if (count<=0||count>1000000) return;\n    size_t size=(size_t)count*sizeof(int);\n    int *data=malloc(size);\n    if (!data) return;\n    free(data);\n}"},
    {"id":"CWE190_s2","cwe":"CWE-190","expected":"SAFE","code":"int safe_add(int a, int b) {\n    if ((b>0&&a>INT_MAX-b)||(b<0&&a<INT_MIN-b)) return -1;\n    return a+b;\n}"},
    # CWE-191
    {"id":"CWE191_v1","cwe":"CWE-191","expected":"VULNERABLE","code":"unsigned int decrement(unsigned int val) {\n    return val - 1;\n}"},
    {"id":"CWE191_v2","cwe":"CWE-191","expected":"VULNERABLE","code":"void process(unsigned int len) {\n    unsigned int adj = len - 10;\n    char *buf = malloc(adj);\n    memset(buf,0,adj);\n}"},
    {"id":"CWE191_v3","cwe":"CWE-191","expected":"VULNERABLE","code":"int subtract(int a, int b) {\n    return a - b;\n}"},
    {"id":"CWE191_s1","cwe":"CWE-191","expected":"SAFE","code":"unsigned int safe_dec(unsigned int val) {\n    if (val==0) return 0;\n    return val-1;\n}"},
    {"id":"CWE191_s2","cwe":"CWE-191","expected":"SAFE","code":"void process_safe(unsigned int len) {\n    if (len<10) return;\n    unsigned int adj=len-10;\n    char *buf=malloc(adj+1);\n    if (!buf) return;\n    free(buf);\n}"},
]

print(f"Running pipeline on {len(rq3_samples)} samples...")
all_results = []

for sample in rq3_samples:
    r = run_pipeline(sample["code"], sample_id=sample["id"])
    r["cwe"] = sample["cwe"]
    r["expected"] = sample["expected"]
    all_results.append(r)

# Summary table
print("\n" + "="*65)
print(f"{'ID':15s} {'Expected':12s} {'Predicted':12s} {'Conf':6s} {'Match':5s}")
print("="*65)
correct = 0
for r in all_results:
    pred = r["inference"]["prediction"]
    exp  = r["expected"]
    match = "✓" if pred == exp else "✗"
    correct += pred == exp
    print(f"{r['sample_id']:15s} {exp:12s} {pred:12s} {r['inference']['confidence']:.3f}  {match}")

print("="*65)
print(f"Accuracy: {correct}/{len(all_results)} ({100*correct/len(all_results):.1f}%)")

# Save
with open("rq3_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("Saved to rq3_results.json")

Running pipeline on 20 samples...

[CWE121_v1] VULNERABLE (0.9999) | CWE: CWE-120: Buffer Copy without Checking Size of Input ('Classic Buffer Overflow')
  Tokens: ['];', '16', ');', 'read', ');']
  Explain: The `strcpy` function is used to copy data from the `src` buffer into `dest`. The `dest` buffer is a...

[CWE121_v2] VULNERABLE (0.9999) | CWE: CWE-120: Buffer Copy without Checking Size of Input ('Classic Buffer Overflow')
  Tokens: ['return', '];', ');', 'char', 'process']
  Explain: The `gets()` function is used to read input into the `buf` array without any bounds checking. Since ...

[CWE121_v3] VULNERABLE (0.9999) | CWE: CWE-120: Buffer Copy without Checking Size of Input ('Classic Buffer Overflow')
  Tokens: ['];', '20', ');', ');', 'char']
  Explain: The `result` buffer is allocated with a fixed size of 20 bytes. Both `strcpy` and `strcat` operation...

[CWE121_s1] VULNERABLE (0.9999) | CWE: ?
  Tokens: ['];', "';", '16', ');', 'char']
  Explain: ...

[CWE121_s2] VULNERABLE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(
    "rq3_results.json",
    "/content/drive/MyDrive/rq3_results.json"
)
print("Saved to Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Drive.


In [ ]:
def faithfulness_score(code, top_tokens, n_mask=3):
    """
    Mask top-N attention tokens and measure prediction change.
    Faithfulness = how much confidence drops when key tokens are removed.
    """
    original = predict_with_attention(code)
    original_conf = original["confidence"]
    original_pred = original["prediction"]

    # Mask top tokens by replacing with [UNK]
    masked_code = code
    tokens_masked = 0
    for token in top_tokens[:n_mask]:
        if token in masked_code and len(token) > 1:
            masked_code = masked_code.replace(token, "[UNK]", 1)
            tokens_masked += 1

    if tokens_masked == 0:
        return {"faithfulness": None, "reason": "no tokens masked"}

    masked_result = predict_with_attention(masked_code)
    masked_conf = masked_result["confidence"]

    # Faithfulness = drop in confidence for original prediction
    delta = original_conf - masked_conf
    prediction_changed = masked_result["prediction"] != original_pred

    return {
        "original_prediction": original_pred,
        "original_confidence": round(original_conf, 4),
        "masked_confidence": round(masked_conf, 4),
        "confidence_delta": round(delta, 4),
        "prediction_changed": prediction_changed,
        "tokens_masked": tokens_masked
    }

# Run faithfulness on all 20 samples
print("Running faithfulness experiment...")
print(f"{'ID':15s} {'Delta':8s} {'Pred changed':12s} {'Tokens masked'}")
print("-" * 55)

faithfulness_results = []
for r in all_results:
    top_toks = r["inference"]["top_attention_tokens"]
    faith = faithfulness_score(r["code"], top_toks, n_mask=3)
    faith["sample_id"] = r["sample_id"]
    faith["expected"] = r["expected"]
    faithfulness_results.append(faith)

    delta_str = f"{faith.get('confidence_delta', 0):.4f}" if faith.get('confidence_delta') is not None else "N/A"
    changed = str(faith.get('prediction_changed', 'N/A'))
    masked = str(faith.get('tokens_masked', 0))
    print(f"{r['sample_id']:15s} {delta_str:8s} {changed:12s} {masked}")

# Summary
valid = [f for f in faithfulness_results if f.get('confidence_delta') is not None]
if valid:
    avg_delta = sum(f['confidence_delta'] for f in valid) / len(valid)
    pred_changed = sum(1 for f in valid if f.get('prediction_changed'))
    print(f"\nAverage confidence delta: {avg_delta:.4f}")
    print(f"Prediction changed after masking: {pred_changed}/{len(valid)} samples")

# Save
with open("faithfulness_results.json", "w") as f:
    json.dump(faithfulness_results, f, indent=2)
print("Saved to faithfulness_results.json")

Running faithfulness experiment...
ID              Delta    Pred changed Tokens masked
-------------------------------------------------------
CWE121_v1       0.0000   False        3
CWE121_v2       0.0000   False        3
CWE121_v3       0.0000   False        3
CWE121_s1       0.0000   False        3
CWE121_s2       0.0000   False        3
CWE78_v1        0.0000   False        3
CWE78_v2        0.0000   False        3
CWE78_v3        0.0001   False        3
CWE78_s1        0.0000   False        3
CWE78_s2        0.0000   False        3
CWE190_v1       0.0000   False        3
CWE190_v2       0.0000   False        3
CWE190_v3       0.0000   False        3
CWE190_s1       0.0000   False        3
CWE190_s2       0.0000   False        3
CWE191_v1       0.0000   False        3
CWE191_v2       0.0000   False        3
CWE191_v3       0.0000   False        3
CWE191_s1       0.0000   False        3
CWE191_s2       0.0000   False        3

Average confidence delta: 0.0000
Prediction changed afte

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, os

save_dir = "/content/drive/MyDrive/rq3_outputs"
os.makedirs(save_dir, exist_ok=True)

for fname in ["rq3_results.json", "faithfulness_results.json"]:
    shutil.copy(fname, f"{save_dir}/{fname}")
    print(f"Saved {fname} to Drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved rq3_results.json to Drive
Saved faithfulness_results.json to Drive


In [ ]:
def faithfulness_aggressive(code, top_tokens, n_mask=5):
    original = predict_with_attention(code)

    # Try masking more tokens
    masked_code = code
    for token in top_tokens[:n_mask]:
        if token in masked_code and len(token) > 1:
            masked_code = masked_code.replace(token, " ", 1)

    # Also try completely emptying the function body
    empty_code = "void func() { return; }"

    masked_result = predict_with_attention(masked_code)
    empty_result = predict_with_attention(empty_code)

    return {
        "original_conf": round(original["confidence"], 4),
        "masked_conf":   round(masked_result["confidence"], 4),
        "empty_conf":    round(empty_result["confidence"], 4),
        "delta_masked":  round(original["confidence"] - masked_result["confidence"], 4),
        "delta_empty":   round(original["confidence"] - empty_result["confidence"], 4),
    }

# Test on first three samples
for r in all_results[:3]:
    faith = faithfulness_aggressive(
        r["code"],
        r["inference"]["top_attention_tokens"]
    )
    print(f"{r['sample_id']:15s} original={faith['original_conf']} masked={faith['masked_conf']} empty={faith['empty_conf']} delta_empty={faith['delta_empty']}")

CWE121_v1       original=0.9999 masked=0.9999 empty=0.9294 delta_empty=0.0705
CWE121_v2       original=0.9999 masked=0.9999 empty=0.9294 delta_empty=0.0705
CWE121_v3       original=0.9999 masked=0.9999 empty=0.9294 delta_empty=0.0705
